# Opening the Vault

Counts the DSAI Vault course files and writes the question split used by the study assistant.

The assistant answers from retrieved notes. If the prompt cannot fix notation and answer format, a fine-tuned copy of the model is trained on `dev` only, so nothing from `test` is in its weights. The split also separates questions we may read while writing prompts (`dev`) from questions we leave unread until evaluation (`test`). The notes themselves stay in the index either way. Holding a topic's notes out would make its questions unanswerable, which is a different test and is covered by `data/unanswerable-questions.yaml`.

A group is a topic. MAI3 is the exception: every card there is tagged `measure`, so MAI3 is grouped by file, and a deck named `*-proofs` stays with its parent deck.

In [1]:
from __future__ import annotations

import random
from collections import Counter, defaultdict
from pathlib import Path

import yaml

ROOT = Path("..").resolve()
COURSES = ROOT / "courses"
SPLIT_PATH = ROOT / "docs" / "held-out-split.yaml"
SEED = 2026
TARGET = 0.20
CAP = 0.40


def load(path: Path):
    return yaml.safe_load(path.read_text(encoding="utf-8")) or {}


def course_dirs():
    return [path.parent for path in sorted(COURSES.glob("semester-*/*/course.yaml"))]


def formula_counts(data) -> tuple[int, int]:
    latex = plain = 0
    for section in data.get("sections") or []:
        for row in section.get("rows") or []:
            formula = row.get("formula") or ""
            if "```text" in formula:
                plain += 1
            elif formula.strip():
                latex += 1
    return latex, plain


def glossary_count(data) -> int:
    if isinstance(data, list):
        return len(data)
    for key in ("terms", "entries", "glossary"):
        if isinstance(data.get(key), list):
            return len(data[key])
    return 0


def group_for(slug: str, kind: str, topic: str | None, file_stem: str) -> str:
    stem = file_stem[: -len("-proofs")] if file_stem.endswith("-proofs") else file_stem
    if slug == "mai3":
        return f"file:{stem}"
    if topic:
        return f"topic:{topic}"
    return f"file:{stem}"


def collect_items(course_dir: Path) -> list[dict]:
    content = course_dir / "content"
    slug = course_dir.name
    items = []

    def add(kind, record, path, topic):
        items.append({
            "id": record.get("id"),
            "kind": kind,
            "topic": topic,
            "group": group_for(slug, kind, topic, path.stem),
            "file": path.relative_to(course_dir).as_posix(),
        })

    decks = content / "decks"
    if decks.exists():
        for path in sorted(decks.glob("*.yaml")):
            data = load(path)
            for card in data.get("cards") or []:
                add("card", card, path, card.get("topic") or data.get("topic"))
    problems = content / "problems"
    if problems.exists():
        for path in sorted(problems.glob("*.yaml")):
            for question in load(path).get("questions") or []:
                add("problem", question, path, question.get("topic"))
    exams = content / "exams"
    if exams.exists():
        for path in sorted(exams.glob("*/*.yaml")):
            for question in load(path).get("questions") or []:
                add("exam", question, path, question.get("topic"))
    return items


def answerless(course_dir: Path) -> Counter:
    """Cards with no answer, and problems with no solution."""
    missing = Counter()
    content = course_dir / "content"
    for path in sorted((content / "decks").glob("*.yaml")) if (content / "decks").exists() else []:
        for card in load(path).get("cards") or []:
            if not (card.get("a") or card.get("steps") or card.get("cloze")):
                missing["cards without an answer"] += 1
    for path in sorted((content / "problems").glob("*.yaml")) if (content / "problems").exists() else []:
        for question in load(path).get("questions") or []:
            if question.get("type") in ("mcq", "multi", "tf"):
                continue
            if not any(question.get(key) for key in ("solution", "steps", "parts", "answer", "choices")):
                missing["problems without a solution"] += 1
    return missing


## What is in the files

In [2]:
rows = []
missing = Counter()
prompts = Counter()
topic_tags = {}
declared_topics = {}

for course_dir in course_dirs():
    meta = load(course_dir / "course.yaml")
    content = course_dir / "content"
    slug = course_dir.name
    declared_topics[slug] = [topic.get("id") for topic in meta.get("topics") or []]
    if not content.exists():
        rows.append({"course": slug, "code": meta.get("code"), "docs": 0, "words": 0,
                     "cards": 0, "problems": 0, "exams": 0, "glossary": 0,
                     "formulas_latex": 0, "formulas_plain": 0})
        continue

    documents = list(content.rglob("*.md"))
    words = sum(len(path.read_text(encoding="utf-8").split()) for path in documents)
    items = collect_items(course_dir)
    glossary = latex = plain = 0
    for path in content.rglob("*.yaml"):
        if path.name == "glossary.yaml":
            glossary += glossary_count(load(path))
        elif path.name == "formulas.yaml":
            latex_n, plain_n = formula_counts(load(path))
            latex += latex_n
            plain += plain_n
    counts = Counter(item["kind"] for item in items)
    topic_tags[slug] = Counter(item["topic"] or "(none)" for item in items if item["kind"] == "card")
    for path in (content / "decks").glob("*.yaml") if (content / "decks").exists() else []:
        for card in load(path).get("cards") or []:
            text = (card.get("q") or card.get("cloze") or "").strip().lower()
            if text:
                prompts[(slug, text)] += 1
    missing += answerless(course_dir)
    rows.append({
        "course": slug,
        "code": meta.get("code"),
        "docs": len(documents),
        "words": words,
        "cards": counts["card"],
        "problems": counts["problem"],
        "exams": counts["exam"],
        "glossary": glossary,
        "formulas_latex": latex,
        "formulas_plain": plain,
    })

written = [row for row in rows if row["docs"] or row["cards"]]
header = f"{'course':8}{'docs':>6}{'words':>8}{'cards':>7}{'probs':>7}{'exams':>7}{'gloss':>7}{'latex':>7}{'plain':>7}"
print(header)
print("-" * len(header))
for row in written:
    print(
        f"{row['course']:8}{row['docs']:6}{row['words']:8}{row['cards']:7}"
        f"{row['problems']:7}{row['exams']:7}{row['glossary']:7}"
        f"{row['formulas_latex']:7}{row['formulas_plain']:7}"
    )
totals = {key: sum(row[key] for row in written) for key in written[0] if key not in ("course", "code")}
print("-" * len(header))
print(
    f"{'TOTAL':8}{totals['docs']:6}{totals['words']:8}{totals['cards']:7}"
    f"{totals['problems']:7}{totals['exams']:7}{totals['glossary']:7}"
    f"{totals['formulas_latex']:7}{totals['formulas_plain']:7}"
)
print()
print(f"Courses with written content: {len(written)} of {len(rows)}")
print(f"Empty courses: {len(rows) - len(written)}")


course    docs   words  cards  probs  exams  gloss  latex  plain
----------------------------------------------------------------
ads2        11   13885    111      0      0      0     24      0
mai3         7   18476    129     33      0     30     33      0
mlst        15   13204     71      7     25      0     21      0
cda         11    6647    133     14     16      0      0     53
mlpr         4    4611     38      5     12     15     11      0
mlut         2    1805      8      0      0     29     32      0
aai          2    3355     23      0      0      0      0      0
dms          4    8002     75     55      0     28     15      0
lugd         8   17053    143     25      0     63     28      0
rai          2    4047     40      0      0     34      0      0
rf           5   14891     99     29      0     52     35      0
----------------------------------------------------------------
TOTAL       71  105976    870    168     53    251    199     53

Courses with written con

## Data-quality problems

Two problems are visible without reading the prose. MAI3 declares several topics, but every card carries the same tag, so a split on that tag would not split anything. CDA stores its formulas as plain text inside ` ```text ` fences. Every other course that has formulas uses LaTeX. That matters for the "wrong notation" error category: the corpus itself is inconsistent about what correct notation looks like.

In [3]:
print("MAI3 declared topics:", ", ".join(declared_topics["mai3"]))
print("MAI3 card tags:")
for tag, count in topic_tags["mai3"].most_common():
    print(f"  {tag}: {count}")

print()
print("Formula notation:")
for row in written:
    if row["formulas_latex"] or row["formulas_plain"]:
        print(f"  {row['course']:6} latex {row['formulas_latex']:3}   plain text {row['formulas_plain']:3}")

sample = load(COURSES / "semester-4" / "cda" / "content" / "reference" / "formulas.yaml")
first = sample["sections"][0]["rows"][0]
print()
print("First CDA formula, as stored:")
print(first["name"])
print(first["formula"].strip())

print()
print("Missing targets:", dict(missing) or "none")
duplicate_prompts = sum(count - 1 for count in prompts.values() if count > 1)
print(f"Repeated card prompts: {duplicate_prompts}")
print()
print("Target: present. Cards have an answer, a derivation, or a cloze. Problems have a solution, parts, or a final answer.")


MAI3 declared topics: measure, probability, functional, rkhs, matrices, lowrank, optimization, integrals
MAI3 card tags:
  measure: 129

Formula notation:
  ads2   latex  24   plain text   0
  mai3   latex  33   plain text   0
  mlst   latex  21   plain text   0
  cda    latex   0   plain text  53
  mlpr   latex  11   plain text   0
  mlut   latex  32   plain text   0
  dms    latex  15   plain text   0
  lugd   latex  28   plain text   0
  rf     latex  35   plain text   0

First CDA formula, as stored:
k-means objective (inertia)
```text
Σᵢ Σ_(x∈Cᵢ) ‖x − μᵢ‖²
```

Missing targets: none
Repeated card prompts: 1

Target: present. Cards have an answer, a derivation, or a cloze. Problems have a solution, parts, or a final answer.


## Question split

Within each course, groups are shuffled with seed `2026` and taken into the test side until it holds about 20% of the questions. At least one group stays on the dev side. A group that would push the test side past 40% is skipped in favour of a smaller one. Courses with only one group cannot be split by topic. Those use an item split, and the record says so.

In [4]:
def assign(items: list[dict], seed: str) -> dict:
    by_group = defaultdict(list)
    for item in items:
        by_group[item["group"]].append(item)
    keys = sorted(by_group)
    rng = random.Random(seed)

    if len(keys) <= 1:
        ordered = sorted(items, key=lambda item: (item["file"], item["id"] or ""))
        rng.shuffle(ordered)
        n_test = round(TARGET * len(ordered))
        n_test = min(max(n_test, 1), len(ordered) - 1) if len(ordered) > 1 else 0
        return {
            "method": "item",
            "note": "One group only, so whole topics cannot be held out.",
            "test_groups": [],
            "dev_groups": keys,
            "test_items": ordered[:n_test],
            "dev_items": ordered[n_test:],
        }

    order = keys[:]
    rng.shuffle(order)
    test_groups = []
    held = 0
    total = len(items)
    for key in order:
        size = len(by_group[key])
        if test_groups and held / total >= TARGET:
            break
        if len(test_groups) + 1 == len(keys):
            break
        if held and (held + size) / total > CAP:
            continue
        test_groups.append(key)
        held += size

    test_items, dev_items = [], []
    for key, group_items in by_group.items():
        (test_items if key in test_groups else dev_items).extend(group_items)
    return {
        "method": "group",
        "note": "Whole groups are on one side.",
        "test_groups": sorted(test_groups),
        "dev_groups": sorted(key for key in keys if key not in test_groups),
        "test_items": test_items,
        "dev_items": dev_items,
    }


def pack(item: dict) -> dict:
    return {key: item[key] for key in ("id", "kind", "group", "file")}


split = {
    "seed": SEED,
    "target_fraction": TARGET,
    "index": (
        "Every note stays retrievable. dev and test are question sets. "
        "dev may be read while writing prompts. test is unread until evaluation."
    ),
    "grouping": (
        "topic tag, except MAI3, which is grouped by file because every card is tagged measure. "
        "A deck ending in -proofs is grouped with its parent deck."
    ),
    "courses": {},
}

print(f"{'course':8}{'method':8}{'items':>7}{'dev':>6}{'test':>6}{'test%':>7}  test groups")
for course_dir in course_dirs():
    if not (course_dir / "content").exists():
        continue
    items = collect_items(course_dir)
    if not items:
        continue
    assigned = assign(items, f"{SEED}-{course_dir.name}")
    fraction = len(assigned["test_items"]) / len(items)
    split["courses"][course_dir.name] = {
        "method": assigned["method"],
        "note": assigned["note"],
        "n_items": len(items),
        "n_dev": len(assigned["dev_items"]),
        "n_test": len(assigned["test_items"]),
        "test_groups": assigned["test_groups"],
        "dev_groups": assigned["dev_groups"],
        "test_items": [pack(item) for item in assigned["test_items"]],
        "dev_items": [pack(item) for item in assigned["dev_items"]],
    }
    groups = ", ".join(assigned["test_groups"]) or "(item split)"
    print(
        f"{course_dir.name:8}{assigned['method']:8}{len(items):7}"
        f"{len(assigned['dev_items']):6}{len(assigned['test_items']):6}{fraction:7.0%}  {groups}"
    )

SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
header = (
    "# Question split for the study assistant.\n"
    "# Regenerated by notebooks/01_open_vault.ipynb. Do not edit by hand.\n"
)
body = yaml.safe_dump(split, sort_keys=False, allow_unicode=True, width=100)
SPLIT_PATH.write_text(header + body, encoding="utf-8")
print()
print(f"Wrote {SPLIT_PATH.relative_to(ROOT)}")


course  method    items   dev  test  test%  test groups
ads2    group       111    76    35    32%  topic:graph-algorithms, topic:graph-search, topic:randomized
mai3    group       162   117    45    28%  file:lebesgue-integral
mlst    group       103    81    22    21%  topic:classifiers, topic:dimred, topic:nlp, topic:svm
cda     group       163   127    36    22%  topic:ens, topic:stats, topic:streams


mlpr    group        55    34    21    38%  topic:features, topic:models
mlut    item          8     6     2    25%  (item split)
aai     group        23    17     6    26%  topic:seminar
dms     group       130    90    40    31%  file:lab-3-preparation, topic:dynamic-systems
lugd    group       168   127    41    24%  file:cf-practice, topic:intro
rai     group        40    31     9    22%  topic:foundations
rf      group       128    79    49    38%  file:introduction-practice, topic:intro

Wrote docs\held-out-split.yaml
